# Notebook 07: ONNX Export & Optimization

This notebook exports the trained YOLOv8 model to **ONNX format** for runtime-agnostic inference, applies **INT8 quantization**, and benchmarks the accuracy/speed tradeoffs.

**Pipeline:**
1. Export YOLOv8 → ONNX (dynamic axes, simplified)
2. Validate ONNX outputs match PyTorch
3. Apply INT8 dynamic quantization
4. Benchmark: PyTorch vs ONNX vs Quantized ONNX
5. Quality gate: mAP drop ≤2%, recall drop ≤1%

In [ ]:
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

PROJECT_ROOT = Path(".").resolve().parent
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data"
YOLO_DIR = DATA_DIR / "pcb-yolo"
DATASET_YAML = YOLO_DIR / "dataset.yaml"

CLASS_NAMES = {
    0: "missing_hole", 1: "mouse_bite", 2: "open_circuit",
    3: "short", 4: "spur", 5: "spurious_copper",
}

pt_model_path = MODELS_DIR / "yolov8_best.pt"
print(f"PyTorch model: {pt_model_path}")

## 1. Export YOLOv8 to ONNX

In [ ]:
if pt_model_path.exists():
    model = YOLO(str(pt_model_path))
    onnx_path = model.export(format="onnx", dynamic=True, simplify=True)
    onnx_path = Path(onnx_path)

    pt_size = pt_model_path.stat().st_size / 1024 / 1024
    onnx_size = onnx_path.stat().st_size / 1024 / 1024
    print(f"PyTorch model: {pt_size:.1f} MB")
    print(f"ONNX model:    {onnx_size:.1f} MB")
    print(f"ONNX saved to: {onnx_path}")
else:
    print("PyTorch model not found — run Notebook 03 first")

## 2. Validate ONNX Model

Verify that ONNX inference produces the same detections as PyTorch.

In [ ]:
# Compare PyTorch vs ONNX predictions on test images
test_img_dir = YOLO_DIR / "images" / "test"
test_images = sorted(test_img_dir.glob("*"))[:10]

if pt_model_path.exists() and onnx_path.exists() and test_images:
    pt_model = YOLO(str(pt_model_path))
    onnx_model = YOLO(str(onnx_path))

    match_count = 0
    for img_path in test_images:
        pt_results = pt_model.predict(str(img_path), verbose=False)
        onnx_results = onnx_model.predict(str(img_path), verbose=False)

        pt_boxes = len(pt_results[0].boxes)
        onnx_boxes = len(onnx_results[0].boxes)

        # Check class agreement
        pt_classes = sorted([int(b.cls[0]) for b in pt_results[0].boxes])
        onnx_classes = sorted([int(b.cls[0]) for b in onnx_results[0].boxes])

        match = pt_classes == onnx_classes
        if match: match_count += 1
        status = 'MATCH' if match else 'DIFF'
        print(f"  {img_path.stem}: PT={pt_boxes} boxes, ONNX={onnx_boxes} boxes [{status}]")

    print(f"\nClass agreement: {match_count}/{len(test_images)} images")
else:
    print("Models or test images not available")

## 3. INT8 Quantization

In [ ]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

if onnx_path.exists():
    quantized_path = MODELS_DIR / "yolov8_best_int8.onnx"
    quantize_dynamic(
        str(onnx_path),
        str(quantized_path),
        weight_type=QuantType.QUInt8,
    )

    q_size = quantized_path.stat().st_size / 1024 / 1024
    print(f"ONNX (fp32):     {onnx_size:.1f} MB")
    print(f"ONNX (int8):     {q_size:.1f} MB")
    print(f"Compression:     {onnx_size/q_size:.2f}x")
else:
    print("ONNX model not found — run export cell first")

## 4. Benchmark Comparison

Measure inference time, model size, and accuracy for all three variants.

In [ ]:
def benchmark_model(model_path, test_images, n_warmup=5, n_runs=50):
    """Benchmark a YOLO model's inference time."""
    model = YOLO(str(model_path))

    # Warmup
    for img_path in test_images[:n_warmup]:
        model.predict(str(img_path), verbose=False)

    # Timed runs
    times = []
    img_list = list(test_images)
    for i in range(n_runs):
        img_path = img_list[i % len(img_list)]
        start = time.time()
        model.predict(str(img_path), verbose=False)
        times.append((time.time() - start) * 1000)

    return np.mean(times), np.std(times)

models_to_test = [
    ("PyTorch (fp32)", pt_model_path),
    ("ONNX (fp32)", onnx_path),
]
if quantized_path.exists():
    models_to_test.append(("ONNX (int8)", quantized_path))

benchmark_results = []
for name, path in models_to_test:
    if path.exists():
        mean_ms, std_ms = benchmark_model(path, test_images)
        size_mb = path.stat().st_size / 1024 / 1024
        benchmark_results.append({
            "Model": name,
            "Size (MB)": f"{size_mb:.1f}",
            "Inference (ms)": f"{mean_ms:.1f} +/- {std_ms:.1f}",
        })
        print(f"{name}: {mean_ms:.1f} ms/image ({size_mb:.1f} MB)")

bench_df = pd.DataFrame(benchmark_results)
print("\n" + bench_df.to_string(index=False))

In [ ]:
# Run validation to get mAP for each variant
val_results = {}
for name, path in models_to_test:
    if path.exists():
        m = YOLO(str(path))
        r = m.val(data=str(DATASET_YAML), split="test", verbose=False)
        val_results[name] = {
            "mAP@0.5": r.box.map50,
            "mAP@0.5:0.95": r.box.map,
            "Recall": r.box.mr,
            "Precision": r.box.mp,
        }
        print(f"{name}: mAP@0.5={r.box.map50:.4f}, Recall={r.box.mr:.4f}")

val_df = pd.DataFrame(val_results).T
print("\n" + val_df.to_string())

## 5. Quality Gate Check

Verify quantization meets thresholds:
- mAP drop ≤ 2%
- Recall drop ≤ 1%

In [ ]:
if "PyTorch (fp32)" in val_results and "ONNX (int8)" in val_results:
    pt = val_results["PyTorch (fp32)"]
    q = val_results["ONNX (int8)"]

    map_drop = (pt["mAP@0.5"] - q["mAP@0.5"]) * 100
    recall_drop = (pt["Recall"] - q["Recall"]) * 100

    print("=== Quality Gate ===")
    map_status = "PASS" if map_drop <= 2.0 else "FAIL"
    recall_status = "PASS" if recall_drop <= 1.0 else "FAIL"

    print(f"mAP@0.5 drop:  {map_drop:.2f}% (threshold: <=2.0%) [{map_status}]")
    print(f"Recall drop:   {recall_drop:.2f}% (threshold: <=1.0%) [{recall_status}]")

    if map_status == "FAIL" or recall_status == "FAIL":
        print("\nRecommendation: Use full-precision ONNX model for deployment")
    else:
        print("\nINT8 quantized model passes quality gates — safe for deployment")
elif "PyTorch (fp32)" in val_results and "ONNX (fp32)" in val_results:
    pt = val_results["PyTorch (fp32)"]
    onnx = val_results["ONNX (fp32)"]
    print("ONNX fp32 vs PyTorch:")
    print(f"  mAP@0.5 diff: {abs(pt['mAP@0.5'] - onnx['mAP@0.5'])*100:.2f}%")
    print(f"  Recall diff:  {abs(pt['Recall'] - onnx['Recall'])*100:.2f}%")
else:
    print("Validation results not available for comparison")

## 6. Accuracy vs Speed Tradeoff

In [ ]:
# Scatter plot: mAP vs inference time
if val_results and benchmark_results:
    fig, ax = plt.subplots(figsize=(8, 6))

    for br in benchmark_results:
        name = br["Model"]
        if name in val_results:
            ms = float(br["Inference (ms)"].split("+/-")[0].strip())
            mAP = val_results[name]["mAP@0.5"]
            size = float(br["Size (MB)"])
            ax.scatter(ms, mAP, s=size * 20, alpha=0.7, zorder=5)
            ax.annotate(f"{name}\n({size:.0f} MB)",
                       (ms, mAP), textcoords="offset points",
                       xytext=(10, 10), fontsize=9)

    ax.set_xlabel("Inference Time (ms)")
    ax.set_ylabel("mAP@0.5")
    ax.set_title("Accuracy vs Speed Tradeoff")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Run benchmark and validation cells first")